<!-- Copyright (c) 2026 João Frazão. SPDX-License-Identifier: Apache-2.0 -->
# 2. Incerteza, sensibilidade e verificação probabilística

Usa um **modelo genérico** (`api.function_model`): qualquer função Python passa a ser um modelo de sistema com requisitos rastreáveis. Aqui a função de Ishigami (Ishigami & Homma 1990, a = 7, b = 0,1), caso de referência da análise de sensibilidade global, com entradas uniformes em [−π, π].

* propagação Monte Carlo/LHS com P5/P50/P95;
* P(passa) de um requisito com intervalo de Wilson e estado probabilístico;
* índices de Sobol' comparados com os valores analíticos.

Definições baratas: ~1 s.

In [ ]:
import os, tempfile
from pathlib import Path
import matplotlib
matplotlib.use("Agg")  # sem ecrã (CI); no Jupyter as figuras aparecem na mesma com display
import matplotlib.pyplot as plt
from acdt import api

# raiz do repositório (o notebook corre em examples/notebooks ou na raiz)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
print(api.version_info())
import math

In [ ]:
A, B = 7.0, 0.1

def ishigami(p):
    return {"Y": math.sin(p["x1"]) + A * math.sin(p["x2"]) ** 2 + B * p["x3"] ** 4 * math.sin(p["x1"])}

model = api.function_model(
    ishigami, {"x1": 0.0, "x2": 0.0, "x3": 0.0},
    requirements=[{"id": "R-Y", "metric": "Y", "comparison": "max", "threshold": 10.0, "text": "Y ≤ 10"}],
    name="Ishigami")
uncertainty = {
    "n": 1000, "seed": 1, "method": "lhs",
    "target_probability": 0.9, "target_source": "exemplo do notebook (PRESSUPOSTO)",
    "inputs": [{"path": f"x{i}", "dist": "uniform", "low": -math.pi, "high": math.pi,
                "source": "Ishigami & Homma 1990"} for i in (1, 2, 3)],
    "sobol": {"n": 4096},
}
out = Path(tempfile.mkdtemp(prefix="acdt_nb2_"))
r = api.uq_study(uncertainty, model, out)
sm = r["propagation"]["summaries"]["Y"]
print(f"Y: P5 {sm['p5']:.3f}  P50 {sm['p50']:.3f}  P95 {sm['p95']:.3f}")

## Verificação probabilística do requisito

In [ ]:
q = r["propagation"]["requirements"]["R-Y"]
print(q["estado_probabilistico"], "|", q["nota"])

## Sobol' contra o analítico

Variâncias analíticas da função de Ishigami (Saltelli et al., *Global Sensitivity Analysis: The Primer*, 2008, §4): V = a²/8 + bπ⁴/5 + b²π⁸/18 + 1/2, V₁ = (1 + bπ⁴/5)²/2, V₂ = a²/8, V₃ = 0, V₁₃ = b²π⁸(1/18 − 1/50).

In [ ]:
pi = math.pi
V = A**2 / 8 + B * pi**4 / 5 + B**2 * pi**8 / 18 + 0.5
V1, V2, V3 = 0.5 * (1 + B * pi**4 / 5) ** 2, A**2 / 8, 0.0
V13 = B**2 * pi**8 * (1 / 18 - 1 / 50)
analytic = {"x1": (V1 / V, (V1 + V13) / V), "x2": (V2 / V, V2 / V), "x3": (V3 / V, V13 / V)}
idx = {e["variable"]: e for e in r["sobol"]["metrics"]["Y"]["indices"]}
print(f"{'':4} {'S1':>7} {'S1 anal.':>9} {'ST':>7} {'ST anal.':>9}")
for v, (s1, st) in analytic.items():
    print(f"{v:4} {idx[v]['S1']:7.3f} {s1:9.3f} {idx[v]['ST']:7.3f} {st:9.3f}")
assert all(abs(idx[v]["S1"] - analytic[v][0]) < 0.05 for v in analytic)  # verificação do exemplo

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
xs = range(3)
ax.bar([x - 0.2 for x in xs], [idx[v]["S1"] for v in analytic], 0.4, label="S1 (acdt)")
ax.bar([x + 0.2 for x in xs], [idx[v]["ST"] for v in analytic], 0.4, label="ST (acdt)")
ax.scatter(xs, [analytic[v][0] for v in analytic], marker="_", s=400, c="k", label="analítico")
ax.scatter(xs, [analytic[v][1] for v in analytic], marker="_", s=400, c="r")
ax.set_xticks(list(xs), list(analytic)); ax.legend(); ax.set_title("Ishigami: índices de Sobol'")
plt.show()
print("ficheiros:", sorted(p.name for p in out.iterdir()))